# Persona Vectors: Screening Training Data by Projection

`persona_vectors_6.ipynb` validated *prediction* (a dataset's mean projection onto a
persona vector predicts the shift fine-tuning on it causes) and `persona_vectors_7.ipynb`
validated *prevention* (steering during training). This notebook tests the paper's third
claim -- *screening*: score every training example individually by its projection onto the
persona vector, drop the highest-scoring ones, and fine-tune on the rest.

Uses the same 3,000 `evil/misaligned_2.jsonl` examples as `_7`. Two new fine-tunes, each on
2,100 examples:
- **screened**: drop the 900 (30%) highest-projection examples.
- **random_drop** (control): drop 900 randomly chosen examples instead.

`random_drop` controls for the smaller training set (fewer examples, fewer steps); the
fair test of screening is `screened` vs `random_drop`. `_7`'s unprotected result (trained
on all 3,000) is shown only as a no-filtering reference.

Reuses `_6`'s cached persona vector and `_7`'s cached training subset and helper
functions. Same one-condition-per-kernel-restart structure as `_6`/`_7`, for the same
reason (unsloth globally monkey-patches `transformers` the first time it trains).

**Model**: Qwen/Qwen2.5-7B-Instruct

In [1]:
import os

# Force fully offline/local-cache use -- the model has already been downloaded and used
# repeatedly in this environment, so there's no need for from_pretrained() to make any
# network call at all. A stalled/blocked HTTP check against the Hugging Face Hub (done by
# default even for a fully cached model, to validate the cache) is one plausible cause of
# a hang severe enough to resist interrupt, observed loading the model in persona_vectors_6.ipynb.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Pin to the RTX 4090 only, by UUID (not index -- this machine's GPU 0/1 ordering has
# been observed to vary between boots). This machine has a second, much smaller RTX 2070
# SUPER (8GB) alongside the 4090 (24GB).
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-3185d7f6-fae1-0c3e-25f3-ad3e260d30b8"

import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
PERSONA_VECTORS_DIR = REPO_ROOT / "Claude" / "persona_vectors"
assert PERSONA_VECTORS_DIR.exists(), f"Expected cloned repo at {PERSONA_VECTORS_DIR}"
sys.path.insert(0, str(PERSONA_VECTORS_DIR))

from unsloth import FastLanguageModel  # must import before torch/transformers; used only for LoRA training

import gc
import json
import random
import time
from functools import partial

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from sft import sft_train
from validate import TrainingConfig

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print("Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 09-18 15:02:40 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 09-18 15:02:40 [__init__.py:239] Automatically detected platform cuda.
WARNING 09-18 15:02:40 [cuda.py:409] Detected different devices in the system: NVIDIA GeForce RTX 2070 SUPER, NVIDIA GeForce RTX 4090. Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4090
Imported sft_train, TrainingConfig, FastLanguageModel from the real persona_vectors repo.


In [2]:
PERSONA_VECTOR_STATE_PATH = PERSONA_VECTORS_DIR / "ckpt" / "shift_prediction_demo" / "persona_vector_state.pt"
assert PERSONA_VECTOR_STATE_PATH.exists(), (
    f"No cached persona vector at {PERSONA_VECTOR_STATE_PATH}. Run persona_vectors_6.ipynb first."
)
state = torch.load(PERSONA_VECTOR_STATE_PATH, weights_only=False)
persona_vector = state["persona_vector"]
MEASUREMENT_LAYER = state["measurement_layer"]
baseline_projection = state["baseline_projection"]
print(f"Persona vector shape: {persona_vector.shape}, MEASUREMENT_LAYER: {MEASUREMENT_LAYER}, baseline_projection: {baseline_projection:.4f}")

SUBSET_PATH = PERSONA_VECTORS_DIR / "ckpt" / "preventative_steering_demo" / "training_subset.json"
assert SUBSET_PATH.exists(), f"No cached training subset at {SUBSET_PATH}. Run persona_vectors_7.ipynb first."
with open(SUBSET_PATH) as f:
    training_subset = json.load(f)
print(f"Loaded {len(training_subset)} training examples from {SUBSET_PATH}")

UNPROTECTED_RESULTS_PATH = PERSONA_VECTORS_DIR / "ckpt" / "preventative_steering_demo" / "results.json"

MISALIGNED_2_PATH = PERSONA_VECTORS_DIR / "dataset" / "evil" / "misaligned_2.jsonl"
assert MISALIGNED_2_PATH.exists(), f"Missing {MISALIGNED_2_PATH} -- run persona_vectors_6.ipynb first."

with open(PERSONA_VECTORS_DIR / "data_generation" / "trait_data_eval" / "evil.json") as f:
    EVAL_QUESTIONS = json.load(f)["questions"]
print(f"Eval questions: {len(EVAL_QUESTIONS)}")

Persona vector shape: torch.Size([29, 3584]), MEASUREMENT_LAYER: 20, baseline_projection: -0.2987
Loaded 3000 training examples from /home/rob/PythonEnvironments/PersonaVectors/PersonaVectors/Claude/persona_vectors/ckpt/preventative_steering_demo/training_subset.json
Eval questions: 20
